# Regression Active Learning: What Should We Learn Next?

Active Learning (AL)では最適値ではなく**未知関数やtarget predictionを効率よく学ぶこと**が目的です。このNotebookでは同じGPから、3種類のlearning utilityで実際の観測candidateを生成します。

- `PosteriorVariance`: candidate自身のuncertainty
- `qNegIntegratedPosteriorVariance`: target region全体のuncertainty reduction
- `ExpectedPredictiveInformationGain`: target predictionsについて得るinformation

関連理論:

- [Active Learning Acquisition](../../docs/theory/acquisition/09_active_learning.md)
- [Regression Active Learning guide](../../docs/optimization/regression_active_learning.md)
- [EPIG guide](../../docs/optimization/epig.md)

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.active_learning import qNegIntegratedPosteriorVariance
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from robotorchan.acquisition import (
    ExpectedPredictiveInformationGain,
    PosteriorStd,
    PosteriorVariance,
)
from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Regression problem

観測点を中央付近に多く置き、未観測領域を残します。ALでは関数値が高い場所ではなく、learning objectiveにとって価値のある場所を選びます。

In [ ]:
def latent_function(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * torch.cos(6 * torch.pi * X)


train_X = torch.tensor([[0.08], [0.28], [0.42], [0.50], [0.58], [0.72], [0.92]])
train_Y = latent_function(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

## 3. Posterior uncertainty

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

## 4. Pointwise uncertainty: PosteriorVariance and PosteriorStd

q=1ではvarianceとstandard deviationは単調変換なので、単純なargmaxは同じです。ただしscore scaleが異なるため、他のutilityやpenaltyとの合成では交換可能とは限りません。

In [ ]:
variance_acq = PosteriorVariance(model)
std_acq = PosteriorStd(model)

variance_candidate, variance_value = optimize_acqf(
    acq_function=variance_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
std_candidate, std_value = optimize_acqf(
    acq_function=std_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

print("variance candidate:", variance_candidate.item())
print("std candidate:", std_candidate.item())
print("same candidate:", torch.allclose(variance_candidate, std_candidate, atol=1e-4))

## 5. Candidate set and target set are different concepts

次は「候補自身が不確実か」ではなく「その候補を観測するとtarget regionをどれだけ学べるか」を考えます。ここではprediction qualityが重要なtarget regionを`x∈[0.35, 0.65]`とします。

In [ ]:
target_X = torch.linspace(0.35, 0.65, 25).unsqueeze(-1)
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.scatter(train_X.squeeze(-1), torch.zeros_like(train_X.squeeze(-1)), label="observed")
ax.scatter(
    target_X.squeeze(-1),
    torch.ones_like(target_X.squeeze(-1)),
    marker="|",
    label="target region",
)
ax.set_yticks([0.0, 1.0], labels=["observed", "target"])
ax.set_xlabel("x")
ax.legend()
plt.show()

## 6. Integrated uncertainty reduction: qNIPV

BoTorch native `qNegIntegratedPosteriorVariance`は、candidate batchを観測した後のtarget points上のintegrated posterior varianceを小さくするように選択します。q=2をjointに最適化するため、高variance点を独立に2個選ぶのとは異なります。

In [ ]:
nipv = qNegIntegratedPosteriorVariance(model=model, mc_points=target_X)
nipv_candidates, nipv_value = optimize_acqf(
    acq_function=nipv,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert nipv_candidates.shape == torch.Size([2, 1])
print("qNIPV joint candidates:", nipv_candidates.squeeze(-1))
print("qNIPV value:", nipv_value)

## 7. Predictive information: EPIG

EPIGはcandidate自身のentropyではなく、candidate observationとtarget predictionsのmutual informationを評価します。現在のrobotorchan実装はq=1・single-output・joint Gaussian posterior・finite target setという明示的contractです。

In [ ]:
epig = ExpectedPredictiveInformationGain(
    model=model,
    target_X=target_X,
    observation_noise=0.01,
)
epig_candidate, epig_value = optimize_acqf(
    acq_function=epig,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

assert epig_candidate.shape == torch.Size([1, 1])
print("EPIG candidate:", epig_candidate.item())
print("EPIG value:", epig_value.item())

## 8. Compare the decision outputs

3つのcriterionは異なるlearning questionに答えるため、acquisition valueの絶対値を共通scoreとして比較しません。candidate位置をposteriorとtarget region上で確認します。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), std, label="posterior std")
ax.axvspan(0.35, 0.65, alpha=0.12, label="target region")
ax.axvline(variance_candidate.item(), linestyle="--", label="PosteriorVariance")
for x in nipv_candidates.squeeze(-1):
    ax.axvline(x.item(), linestyle=":", label="qNIPV" if x is nipv_candidates[0] else None)
ax.axvline(epig_candidate.item(), linestyle="-.", label="EPIG")
ax.set_xlabel("x")
ax.set_ylabel("posterior std")
ax.legend()
plt.show()

## 9. Evaluate one AL candidate and update the dataset

代表としてEPIG candidateを観測します。AL loopでもcandidate生成後は実験・simulationでlabel/responseを取得し、training dataへ追加してsurrogateを更新します。

In [ ]:
candidate = epig_candidate
candidate_Y = latent_function(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
fit_gpytorch_mll(updated_model.make_mll())
updated_model.eval()
print("candidate:", candidate)
print("observed response:", candidate_Y)
print("updated n:", updated_X.shape[0])

## 10. Verify uncertainty changes in the target region

In [ ]:
with torch.no_grad():
    before_var = model.posterior(target_X).variance.mean()
    after_var = updated_model.posterior(target_X).variance.mean()
print("mean target variance before:", before_var.item())
print("mean target variance after:", after_var.item())

## 11. Decision semantics

| Acquisition | Learns what? | Candidate contract |
| --- | --- | --- |
| PosteriorVariance / Std | candidate自身のmarginal uncertainty | q=1 |
| qNIPV | target region全体のposterior uncertainty | joint q batch |
| EPIG | target predictionsとのinformation dependence | current implementation q=1 |

ALではまず「どこでprediction qualityが必要か」を決めることが重要です。target setがdeployment distributionを表さなければ、target-aware acquisitionも誤った領域を重視します。

## 12. Epistemic and aleatoric uncertainty

追加観測で減らしたいのは主にepistemic uncertaintyです。観測noiseが大きいだけの場所をpredictive varianceだけで追うと、irreducibleなaleatoric uncertaintyを繰り返し観測する可能性があります。

このNotebookはnoise-free synthetic latent functionを使い、criterion間のdecision semanticsに焦点を当てています。heteroskedastic/noise-modelingは専用Notebookを参照してください。

## 13. Scope boundaries

- BOのEI/UCBはoptimum discoveryが目的であり、このNotebookのlearning utilityとは異なります。
- Straddle / BoundaryVariance / RandomizedStraddleはlevel-set・boundary learningとして別Phaseで扱います。
- classificationのPredictiveEntropy、BALD、Margin等はclassification model contractが整うまで対応済みとはしません。
- EPIGをNGBoostやtree ensembleへ適用できるとは主張しません。joint Gaussian covarianceが必要です。
- qNIPVはBoTorch native APIを直接使い、robotorchan wrapperを追加しません。
- stopping criterionはacquisitionとは別責務です。

## 14. Related notebooks

- `01_single_task_gp.ipynb`: single-objective BO loop
- `16_noise_models.ipynb`: observation-noise semantics
- `26_standard_acquisition.ipynb`: BO acquisition comparison

このNotebookはregression Active Learningのdecision workflowを担当します。